# NB00 · Setup, Python and RDKit Refresher
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/prateekpaul/Cheminformatics/blob/main/notebooks/nb00_setup_and_refresher.ipynb)

**Estimated time:** ~30 min | **Difficulty:** ●○○

A hands-on onboarding guide verifying the local computational cheminformatics environment, introducing fundamental molecular operations in RDKit and Open Babel, manipulating tabular molecular datasets via `PandasTools`, and establishing robust error handling patterns.

## Learning Objectives

By the end of this notebook, you will be able to:
1. **Verify** your computational environment and inspect dependencies (`rdkit`, `openbabel`, `vina`, `meeko`, `mordred`, Java).
2. **Construct and visualize** 2D molecular graphs from SMILES strings using RDKit (`MolFromSmiles`, `MolToSmiles`, `Draw`).
3. **Iterate** across molecular graph elements, extracting atomic properties and bond orders.
4. **Differentiate** between implicit and explicit hydrogens (`AddHs`, `RemoveHs`) and quantify their effect on atomic indices.
5. **Manipulate** chemical datasets in pandas DataFrames with `PandasTools`.
6. **Execute** format conversions and 3D coordinate generation via Open Babel.
7. **Handle** chemical errors gracefully using sanitization flags and logger controls.

## Prerequisites and Links

- Basic Python programming knowledge (functions, dictionaries, list comprehensions, pandas).
- Course context: Lecture 01 (Course Introduction) and Lecture 03 (Cheminformatics Tools).

In [ ]:
import os, sys, random, warnings, shutil, subprocess
import numpy as np
import pandas as pd
from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem, Draw, Descriptors, PandasTools

# Prepend active Python directory to PATH to ensure external CLI tools (obabel, vina) are found
conda_bin = os.path.dirname(sys.executable)
if conda_bin not in os.environ.get("PATH", ""):
    os.environ["PATH"] = conda_bin + os.pathsep + os.environ.get("PATH", "")

SEED = 42
FULL_RUN = False
random.seed(SEED)
np.random.seed(SEED)

# Disable verbose C++ logger warnings initially; we re-enable them in the error handling section
RDLogger.DisableLog("rdApp.*")
warnings.filterwarnings("ignore", category=DeprecationWarning)

print("Environment setup completed successfully.")

## 1. Environment Verification

Before beginning chemical data processing, we verify that our core libraries and command-line tools are operational.

In [ ]:
import rdkit
print(f"RDKit version:       {rdkit.__version__}")

try:
    import openbabel
    from openbabel import openbabel as ob
    print(f"Open Babel (Python): {openbabel.__version__}")
except ImportError:
    print("Open Babel (Python): Not installed")

obabel_path = shutil.which("obabel")
print(f"Open Babel (CLI):    {obabel_path if obabel_path else 'Not found in PATH'}")

vina_path = shutil.which("vina")
print(f"AutoDock Vina CLI:   {vina_path if vina_path else 'Not found in PATH'}")

java_path = shutil.which("java")
print(f"Java Runtime:        {java_path if java_path else 'Not found in PATH (PaDEL requires Java)'}")

try:
    import meeko
    print(f"Meeko version:       {meeko.__version__}")
except ImportError:
    print("Meeko:               Not installed")

## 2. Basic Molecular Manipulation in RDKit

A molecule in RDKit is represented as a `ROMol` (Read-Only Molecule) or `RWMol` (Read-Write Molecule) object. Under the hood, this is a C++ graph object where atoms are nodes and bonds are edges.

In [ ]:
# Parsing a SMILES string
ethanol = Chem.MolFromSmiles("CCO")
aspirin = Chem.MolFromSmiles("CC(=O)Oc1ccccc1C(=O)O")

print(f"Ethanol heavy atom count: {ethanol.GetNumHeavyAtoms()}")
print(f"Aspirin heavy atom count: {aspirin.GetNumHeavyAtoms()}")
print(f"Aspirin canonical SMILES: {Chem.MolToSmiles(aspirin)}")

# Visualizing molecules
Draw.MolsToGridImage([ethanol, aspirin], molsPerRow=2, subImgSize=(300, 200), legends=["Ethanol (CCO)", "Aspirin"])

### Atom and Bond Iteration

Let us inspect the atoms and bonds of aspirin.

In [ ]:
atom_info = []
for atom in aspirin.GetAtoms():
    atom_info.append({
        "Idx": atom.GetIdx(),
        "Symbol": atom.GetSymbol(),
        "AtomicNum": atom.GetAtomicNum(),
        "Degree": atom.GetDegree(),
        "FormalCharge": atom.GetFormalCharge(),
        "Hybridization": str(atom.GetHybridization()),
        "IsAromatic": atom.GetIsAromatic()
    })
df_atoms = pd.DataFrame(atom_info)
print(df_atoms.head(8))

bond_info = []
for bond in aspirin.GetBonds():
    bond_info.append({
        "BondIdx": bond.GetIdx(),
        "BeginAtom": bond.GetBeginAtomIdx(),
        "EndAtom": bond.GetEndAtomIdx(),
        "BondType": str(bond.GetBondType()),
        "IsAromatic": bond.GetIsAromatic()
    })
df_bonds = pd.DataFrame(bond_info)
print(df_bonds.head(6))

> [!NOTE] What to notice
> By default, RDKit creates a **hydrogen-depleted molecular graph**. The hydrogen atoms are implicit and tracked as an integer property (`atom.GetTotalNumHs()`) on each heavy atom. Atom indexing starts at 0.

## 3. Implicit vs. Explicit Hydrogens

In 2D graph manipulation and topological descriptor calculation, hydrogens are usually kept implicit to conserve memory and simplify graph traversal. However, for 3D coordinate generation, docking, and molecular mechanics, **explicit hydrogens are mandatory**.

In [ ]:
mol_depleted = Chem.MolFromSmiles("CCO")
mol_explicit = Chem.AddHs(mol_depleted)

print(f"Heavy atoms only:      {mol_depleted.GetNumAtoms()} atoms, {mol_depleted.GetNumBonds()} bonds")
print(f"With explicit H atoms: {mol_explicit.GetNumAtoms()} atoms, {mol_explicit.GetNumBonds()} bonds")

mol_recovered = Chem.RemoveHs(mol_explicit)
print(f"After RemoveHs:        {mol_recovered.GetNumAtoms()} atoms")

## 4. Molecular Data in Pandas via `PandasTools`

`rdkit.Chem.PandasTools` seamlessly integrates RDKit molecules into pandas DataFrames, rendering 2D structure depictions directly within notebook tables.

In [ ]:
csv_path = "../data/processed/d01_slide_draw_smiles.csv"
if not os.path.exists(csv_path):
    csv_path = "data/processed/d01_slide_draw_smiles.csv"

df = pd.read_csv(csv_path)
PandasTools.AddMoleculeColumnToFrame(df, smilesCol="smiles", molCol="ROMol")
df["HeavyAtoms"] = df["ROMol"].apply(lambda m: m.GetNumHeavyAtoms() if m else 0)
df["MW"] = df["ROMol"].apply(lambda m: round(Descriptors.MolWt(m), 2) if m else 0.0)
df[["id", "name", "smiles", "HeavyAtoms", "MW"]].head(6)

## 5. Open Babel Quick Tour

Open Babel is a versatile chemical toolbox specialized in inter-format translation. We can access Open Babel via its Python bindings or via its command-line interface `obabel`.

In [ ]:
from openbabel import openbabel as ob

obConversion = ob.OBConversion()
in_fmts = obConversion.GetSupportedInputFormat()
out_fmts = obConversion.GetSupportedOutputFormat()

# Note vs slides: The original lecture slide reported 183 formats.
print(f"Open Babel supported input formats:  {len(in_fmts)}")
print(f"Open Babel supported output formats: {len(out_fmts)}")

# 0D to 3D conversion using Open Babel in Python
mol_ob = ob.OBMol()
obConversion.SetInAndOutFormats("smi", "mol2")
obConversion.ReadString(mol_ob, "CCCC")  # n-butane
mol_ob.AddHydrogens()

builder = ob.OBBuilder()
builder.Build(mol_ob)
mol2_text = obConversion.WriteString(mol_ob)

print("Generated MOL2 text (first 5 lines):")
for line in mol2_text.splitlines()[:5]:
    print(line)

## 6. Error Handling and Sanitization Flags

When parsing chemical data from external sources, invalid syntax or chemically nonsensical valences are common. RDKit returns `None` for invalid molecules.

In [ ]:
# Re-enable logger to view C++ warning messages
RDLogger.EnableLog("rdApp.*")

# Invalid SMILES: unclosed ring
bad_mol = Chem.MolFromSmiles("C1CC")
print(f"Parsed 'C1CC' -> {bad_mol}")

# Disabling sanitization to examine problematic intermediate structures (e.g. hypervalent carbon)
raw_mol = Chem.MolFromSmiles("C(C)(C)(C)(C)C", sanitize=False)
print(f"Without sanitization -> {raw_mol} (Atoms: {raw_mol.GetNumAtoms()})")

# Re-disable logger for rest of notebook
RDLogger.DisableLog("rdApp.*")

> [!WARNING] None Propagation Pitfall
> Always filter out `None` values before applying molecular descriptor functions in pandas pipelines (e.g. `df['ROMol'].dropna()`), otherwise `AttributeError: 'NoneType' object has no attribute 'GetNumHeavyAtoms'` will halt execution.

### Exercise 0.1

Write a function `parse_and_validate_smiles(smiles_list: list[str]) -> dict[str, bool]` that parses a list of SMILES strings and returns a dictionary mapping each SMILES to `True` if it is valid (non-None) and `False` otherwise.

In [ ]:
def parse_and_validate_smiles(smiles_list: list[str]) -> dict[str, bool]:
    # YOUR CODE HERE
    raise NotImplementedError("Implement parse_and_validate_smiles")

In [ ]:
# Self-check for Exercise 0.1
test_smiles = ["CCO", "c1ccccc1", "C(=O)O", "C1CC", "CCN(CC)CC"]
val_res = parse_and_validate_smiles(test_smiles)
assert val_res["CCO"] is True
assert val_res["c1ccccc1"] is True
assert val_res["C1CC"] is False
assert len(val_res) == 5
print('✔ Exercise 0.1 passed!')

### Exercise 0.2

Write a function `count_heavy_atoms_and_bonds(smiles: str) -> tuple[int, int]` that returns a tuple of `(num_heavy_atoms, num_bonds)` for the given SMILES string. If the SMILES is invalid, raise a `ValueError`.

In [ ]:
def count_heavy_atoms_and_bonds(smiles: str) -> tuple[int, int]:
    # YOUR CODE HERE
    raise NotImplementedError("Implement count_heavy_atoms_and_bonds")

In [ ]:
# Self-check for Exercise 0.2
assert count_heavy_atoms_and_bonds("CCO") == (3, 2)
assert count_heavy_atoms_and_bonds("CC(=O)Oc1ccccc1C(=O)O") == (13, 13)
assert count_heavy_atoms_and_bonds("c1ccccc1") == (6, 6)
try:
    count_heavy_atoms_and_bonds("InvalidSmi")
    assert False, "Should have raised ValueError"
except ValueError:
    pass
print('✔ Exercise 0.2 passed!')

### Exercise 0.3

Write a function `get_formal_charges(smiles: str) -> dict[int, int]` that returns a dictionary mapping atom indices to their formal charge, for only those atoms having non-zero formal charge.

In [ ]:
def get_formal_charges(smiles: str) -> dict[int, int]:
    # YOUR CODE HERE
    raise NotImplementedError("Implement get_formal_charges")

In [ ]:
# Self-check for Exercise 0.3
fc_salt = get_formal_charges("CC(=O)[O-].[Na+]")
assert set(fc_salt.values()) == {-1, 1}
fc_neutral = get_formal_charges("CCO")
assert len(fc_neutral) == 0
print('✔ Exercise 0.3 passed!')

## Common Pitfalls

1. **`None` Object Propagation:** Attempting to invoke RDKit methods on a `None` result causes unhandled exceptions in data science workflows. Always check `if mol is not None:` or use `.dropna()`.
2. **Implicit Hydrogens in 3D Embedding:** Omitting `Chem.AddHs(mol)` before calling 3D conformation embedding generates hydrogen-less 3D coordinates, distorting steric geometry.
3. **Log Suppression Concealing Bugs:** Disabling `RDLogger` permanently can obscure why a critical SMILES failed to kekulize. Selectively re-enable logging when debugging.

## Summary and Key Takeaways

- RDKit molecules are graph objects where vertices are atoms and edges are covalent bonds.
- Molecular graphs default to hydrogen-depleted representations; explicit hydrogens must be added via `Chem.AddHs` for 3D tasks.
- `PandasTools` enables high-throughput visual inspection of chemical datasets in tabular format.
- Open Babel provides comprehensive format conversion capabilities both as a CLI tool and via Python bindings.
- Chemical error handling requires explicit checking for `None` and an understanding of sanitization stages.

## References

- Landrum, G. *RDKit: Open-source cheminformatics*. https://www.rdkit.org
- O'Boyle, N. M. et al. Open Babel: An open chemical toolbox. *J. Cheminformatics* **2011**, 3, 33.
- Leach, A. R.; Gillet, V. J. *An Introduction to Chemoinformatics*. Springer, 2007.

In [ ]:
# Self-test check
print("✔ NB00 checks passed")